# Environment Setup

In [ ]:
!pip install pydicom

In [ ]:
import sys
import numpy as np
from typing import Tuple, Dict, Any
from pathlib import Path
import matplotlib.pyplot as plt
from skimage.draw import polygon
from scipy.spatial import ConvexHull, QhullError
from scipy.ndimage import binary_closing
import pydicom
from pydicom.uid import generate_uid, ExplicitVRLittleEndian
import os

PRIVATE_CREATOR = "LabBioimmagini v1"
PRIVATE_GROUP = 0x0035

import logging

logger = logging.getLogger(__name__)

The following cell configures your workspace to ensure the notebook can access all necessary data files, regardless of where you are running it.

**What this cell does:**
* **Google Colab:** Automatically detects the environment and adjusts the working directory to the path specified in the `%cd` command.
* **Local Machine:** Prompts you to input the path to your **base data folder** (the folder containing the `lab-0-data` directory).

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount the drive containing the images
    drive.mount('/content/drive')
    
    # List contents (IPython magic command)
    !ls

In [ ]:
if IN_COLAB:
    # Change the working to the current notebook's directory
    %cd /content/drive/MyDrive/bioimages-project-2026/
else:
    # Ask the user to fill in the base data folder path
    base_data_folder = input("Please enter the path to the base data folder path: ")
    os.chdir(base_data_folder)

Verify that the following cell is printing the name of the folder you downloaded from the Google Drive link (e.g., `lab-0-data`). Otherwise, please check the path you provided in the previous cell.

In [ ]:
if IN_COLAB:
    !ls
else:
    files = os.listdir(base_data_folder)
    for f in files:
        print(f)

# Simple Circular Sector Segmentation Example

## Per-cell Code Implementation

This is an example of how you can use per-cell code execution to visually inspect the results of the code implemented, as a sort of "debugging" tool. The code below demonstrates how to segment a circular sector and visualize the results.

In [ ]:
IMAGE_PATH = "./lab-0-data/camus/0001.dcm"

ds = pydicom.dcmread(IMAGE_PATH)
im_array = ds.pixel_array.astype(np.float32)
if im_array.max() > 1.0:
    im_array = im_array / 255.0

print(f"  Shape   : {ds.Rows} x {ds.Columns}")
if hasattr(ds, "PixelSpacing"):
    print(f"  Spacing : {list(ds.PixelSpacing)} mm")
try:
    blk = ds.private_block(PRIVATE_GROUP, PRIVATE_CREATOR)
    for offset, label in [
        (0x01, "dataset"),
        (0x03, "kind"),
        (0x04, "view"),
        (0x05, "phase"),
        (0x06, "quality"),
    ]:
        try:
            print(f"  {label:<10}: {ds[blk.get_tag(offset)].value}")
        except KeyError:
            pass
except KeyError:
    pass

plt.imshow(im_array, cmap="gray")
plt.title(Path(IMAGE_PATH).name)
plt.axis("off")
plt.show()

In [ ]:
im_array.shape

In [ ]:
crop_size = 10
half_size = crop_size // 2

y_coords, x_coords = np.where(im_array != 0)

if (
    len(y_coords) > 0
    and im_array.shape[0] >= crop_size
    and im_array.shape[1] >= crop_size
):
    # Randomly select one non-zero pixel to be the focal point of the crop
    random_idx = np.random.randint(len(y_coords))
    center_y = y_coords[random_idx]
    center_x = x_coords[random_idx]

    max_y, max_x = im_array.shape[:2]

    # Calculate the top-left corner of the crop
    start_y = max(0, min(center_y - half_size, max_y - crop_size))
    start_x = max(0, min(center_x - half_size, max_x - crop_size))

    # Extract the exact 10x10 region
    random_crop = im_array[start_y : start_y + crop_size, start_x : start_x + crop_size]

    plt.imshow(random_crop, cmap="gray")
    plt.title(f"Random {crop_size}x{crop_size} Crop")
    plt.show()
else:
    print(
        "Cannot extract crop: Image is either all zeros or smaller than the requested crop size."
    )

In [ ]:
random_crop, random_crop.shape

In [ ]:
binary = im_array > 0.01
radius = 3
y, x = np.ogrid[-radius : radius + 1, -radius : radius + 1]
se = (x**2 + y**2) <= radius**2
binary = binary_closing(binary, structure=se)

rows, cols = np.nonzero(binary)  # pyright: ignore[reportAssignmentType]
points = np.column_stack([rows, cols])

if len(points) < 3:
    # Degenerate: fewer than 3 points — fall back to the binarized mask.
    mask = binary.astype(bool)
else:
    try:
        hull = ConvexHull(points)
    except QhullError:
        # All points collinear or otherwise degenerate.
        mask = binary.astype(bool)

    hull_vertices = points[hull.vertices]
    rr, cc = polygon(hull_vertices[:, 0], hull_vertices[:, 1], shape=im_array.shape)
    mask = np.zeros(im_array.shape, dtype=bool)
    mask[rr, cc] = True

plt.imshow(mask, cmap="gray")
plt.show()

In [ ]:
_image_path = Path(IMAGE_PATH)
_stem = _image_path.stem  # e.g. "0001"
_output_path = _image_path.parent / f"{_stem}_sector_mask.dcm"

_file_meta = pydicom.Dataset()
_file_meta.MediaStorageSOPClassUID = "1.2.840.10008.5.1.4.1.1.7"
_file_meta.MediaStorageSOPInstanceUID = generate_uid()
_file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

_ds = pydicom.dataset.FileDataset(
    str(_output_path),
    {},
    file_meta=_file_meta,
    preamble=b"\x00" * 128,
)
_ds.SOPClassUID = "1.2.840.10008.5.1.4.1.1.7"
_ds.SOPInstanceUID = _file_meta.MediaStorageSOPInstanceUID
_ds.Modality = "US"
_ds.Rows = mask.shape[0]
_ds.Columns = mask.shape[1]
_ds.SamplesPerPixel = 1
_ds.PhotometricInterpretation = "MONOCHROME2"
_ds.BitsAllocated = 8
_ds.BitsStored = 8
_ds.HighBit = 7
_ds.PixelRepresentation = 0
_ds.PixelData = (mask.astype(np.uint8) * 255).tobytes()
_ds.save_as(str(_output_path), enforce_file_format=True)
print(f"Saved sector mask to {_output_path}")

## Function-based Reorganization for Clean Coding Practices

This section demonstrates how to organize your code into **functions**—a crucial practice for maintainability and readability. 

Instead of relying on per-cell execution, we will refactor the logic for **circular sector segmentation** and **visualization** into dedicated, single-task functions. This approach ensures your code is:
* **Easy to read** and understand at a glance.
* **Simple to test**, modify, and debug.
* **Highly reusable** across different parts of your project.

---

> **⚠️ Important: Intermediate Project Deliverable Guidelines**
>
> This structured, well-commented code format is **mandatory** for your intermediate deliverable **IF** you choose not to submit a separate written report. 
> 
> * **Code-Only Submissions:** Your code must act as your report. As we will cover in theory and practical sessions, your code and comments must clearly explain the *reasoning* behind the specific image transformations you apply to restore image quality.
> * **Separate Report Submissions:** If you submit a dedicated text report for the intermediate stage, this strict code-as-report formatting is optional.
> * **Final Project Deliverable:** This formatting is not required, as your final submission will include a complete project presentation.

In [ ]:
IMAGE_EXAMPLE_PATH = "./lab-0-data/camus/0001.dcm"

_THRESHOLD = 0.01
_CLOSE_RADIUS = 3

In [ ]:
def dicom_load(filepath: str | Path) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Loads a Secondary Capture DICOM and returns the pixel array and metadata.

    Args:
        filepath: Path to the .dcm file.

    Returns:
        - (H, W) float32 array normalised to [0, 1].
        - Metadata dict with standard and LabBioimmagini private-block tags.
    """
    ds = pydicom.dcmread(str(filepath))

    im_array = ds.pixel_array.astype(np.float32)
    if im_array.max() > 1.0:
        im_array = im_array / 255.0

    info: Dict[str, Any] = {
        "rows": ds.Rows,
        "columns": ds.Columns,
    }
    if hasattr(ds, "PixelSpacing"):
        info["spacing_mm"] = [float(v) for v in ds.PixelSpacing]

    try:
        blk = ds.private_block(PRIVATE_GROUP, PRIVATE_CREATOR)
        for offset, key in [
            (0x01, "dataset"),
            (0x03, "kind"),
            (0x04, "view"),
            (0x05, "phase"),
            (0x06, "quality"),
        ]:
            try:
                info[key] = ds[blk.get_tag(offset)].value
            except KeyError:
                pass
    except KeyError:
        pass

    return im_array, info

In [ ]:
def extract_image_crop(im_array: np.ndarray, crop_size: int = 10) -> np.ndarray | None:
    """Extracts a random square crop of a specified size centered around a
    non-zero pixel in the image.

    Args:
      im_array : The input 2D image array from which to extract the crop.
      crop_size : The height and width of the square crop (default is 10).

    Returns:
        The extracted crop array, or None if the image is empty or
        smaller than the requested crop size.
    """
    half_size = crop_size // 2

    # Find coordinates of all non-zero pixels in the image
    y_coords, x_coords = np.where(im_array != 0)

    # Check if there are non-zero pixels and if the image meets size requirements
    if (
        len(y_coords) > 0
        and im_array.shape[0] >= crop_size
        and im_array.shape[1] >= crop_size
    ):
        # Randomly select one non-zero pixel to serve as the focal point
        random_idx = np.random.randint(len(y_coords))
        center_y = y_coords[random_idx]
        center_x = x_coords[random_idx]

        max_y, max_x = im_array.shape[:2]

        # Calculate the top-left corner, ensuring boundaries stay within the image
        start_y = max(0, min(center_y - half_size, max_y - crop_size))
        start_x = max(0, min(center_x - half_size, max_x - crop_size))

        # Extract the crop region
        random_crop = im_array[
            start_y : start_y + crop_size, start_x : start_x + crop_size
        ]

        return random_crop
    else:
        print(
            "Cannot extract crop: Image is either all zeros or smaller than the"
            " requested crop size."
        )
        return None

In [ ]:
def _disk_structuring_element(radius: int) -> np.ndarray:
    """
    Generates a 2D circular (disk) structuring element for morphological operations
    (e.g., dilation, erosion, opening, or closing) in image processing.

    Args:
        radius (int): The radius of the disk in pixels.

    Returns:
        np.ndarray: A 2D boolean array of shape (2*radius + 1, 2*radius + 1)
        where True values represent the pixels inside the disk.
    """
    # Create an open grid of coordinates (y, x) centered at (0, 0).
    # np.ogrid returns sparse 1D arrays that broadcast together to form the 2D grid,
    # making it more memory efficient than np.meshgrid.
    y, x = np.ogrid[-radius : radius + 1, -radius : radius + 1]

    # Apply the standard circle equation (x^2 + y^2 <= r^2) to determine
    # which pixels fall within the specified radius from the center.
    return (x**2 + y**2) <= radius**2

In [ ]:
def compute_sector_mask(image: np.ndarray) -> np.ndarray:
    """
    Compute the sector mask from a float32 image through
    the convex hull of pixels above a small threshold, after robust binarization.
    It always contains any interior zero pixels (dark blood pools, chamber interiors).

    Args:
        image (np.ndarray): 2D float32 image.

    Returns:
        np.ndarray: A boolean array where True indicates pixels inside the ultrasound fan.
    """
    if image.ndim != 2:
        raise ValueError(f"Expected 2-D image, got shape {image.shape}")

    # 1. INITIAL THRESHOLDING
    # Identify all pixels with a signal intensity greater than 0.01.
    # This gives a rough outline of the cone but may contain holes or jagged edges.
    binary = image > _THRESHOLD

    if not binary.any():
        logger.warning("compute_sector_mask: all-zero frame — returning empty mask")
        return np.zeros(image.shape, dtype=bool)

    # 2. MORPHOLOGICAL SMOOTHING
    # Create a small circular structuring element (radius of 3).
    se = _disk_structuring_element(_CLOSE_RADIUS)
    # Apply morphological "closing". This expands the bright areas slightly and
    # then shrinks them back, which bridges small gaps and smooths frayed edges.
    binary = binary_closing(binary, structure=se)

    # 3. COORDINATE EXTRACTION
    # Get the row (y) and column (x) coordinates of all the True (active) pixels.
    rows, cols = np.nonzero(binary)  # pyright: ignore[reportAssignmentType]
    # Stack them into a list of [y, x] points for the convex hull algorithm.
    points = np.column_stack([rows, cols])

    # 4. CONVEX HULL GENERATION (The "Rubber Band")
    if len(points) < 3:
        # Degenerate case: You need at least 3 points to form a 2D shape.
        # If the image is basically empty, just use the raw threshold mask.
        mask = binary.astype(bool)
        return mask

    try:
        # Calculate the convex hull. Imagine stretching a rubber band around
        # the outermost 'points'. This ignores internal holes (dropouts)
        # and unifies disconnected blobs into one solid sector cone.
        hull = ConvexHull(points)
    except QhullError:
        # Fallback: If all points form a straight line (collinear) or are
        # mathematically degenerate, the hull fails. Fall back to the raw mask.
        mask = binary.astype(bool)
        return mask

    # Extract the specific points that form the outer boundary.
    hull_vertices = points[hull.vertices]
    # Convert those boundary vertices into a filled 2D polygon.
    # 'rr' and 'cc' are the row and column indices of all pixels INSIDE the hull.
    rr, cc = polygon(hull_vertices[:, 0], hull_vertices[:, 1], shape=image.shape)
    # Create a blank, black mask and fill the polygon area with True (white).
    mask = np.zeros(image.shape, dtype=bool)
    mask[rr, cc] = True
    return mask

In [ ]:
def save_sector_mask(mask: np.ndarray, output_path: str | Path) -> None:
    """
    Save a boolean sector mask as a Secondary Capture DICOM file.
    Pixel values are 0 (outside sector) or 255 (inside sector).

    Args:
        mask: 2D boolean array produced by compute_sector_mask.
        output_path: Destination file path (.dcm).
    """
    output_path = Path(output_path)

    file_meta = pydicom.Dataset()
    file_meta.MediaStorageSOPClassUID = "1.2.840.10008.5.1.4.1.1.7"
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

    ds = pydicom.dataset.FileDataset(
        str(output_path),
        {},
        file_meta=file_meta,
        preamble=b"\x00" * 128,
    )
    ds.SOPClassUID = "1.2.840.10008.5.1.4.1.1.7"
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.Modality = "US"
    ds.Rows = mask.shape[0]
    ds.Columns = mask.shape[1]
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    ds.BitsAllocated = 8
    ds.BitsStored = 8
    ds.HighBit = 7
    ds.PixelRepresentation = 0
    ds.PixelData = (mask.astype(np.uint8) * 255).tobytes()
    ds.save_as(str(output_path), enforce_file_format=True)

In [ ]:
image, info = dicom_load(IMAGE_EXAMPLE_PATH)
for key, value in info.items():
    print(f"  {key:<12}: {value}")
plt.imshow(image, cmap="gray")
plt.title(Path(IMAGE_EXAMPLE_PATH).name)
plt.axis("off")
plt.show()

In [ ]:
crop = extract_image_crop(image, crop_size=10)
if crop is not None:
    print(f"Successfully extracted crop with shape: {crop.shape}")
    plt.imshow(crop, cmap="gray")
    plt.title("Random 10x10 Crop")
    plt.axis("off")
    plt.show()

In [ ]:
mask = compute_sector_mask(image)
plt.imshow(mask, cmap="gray")
plt.show()

In [ ]:
image_path = Path(IMAGE_EXAMPLE_PATH)
stem = image_path.name.split(".")[0]
output_path = image_path.parent / f"{stem}_sector_mask.dcm"
save_sector_mask(mask, output_path)
print(f"Saved sector mask to {output_path}")